<a href="https://colab.research.google.com/github/RCDS13/PI-JP-4SEM/blob/main/etl_transito.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
from google.colab import drive
import os
drive.mount('/content/drive', force_remount=True)
!ls "/content/drive/MyDrive/PI_JP"

!pip install rpy2
%load_ext rpy2.ipython

In [ ]:
%%R

library(dplyr)
library(stringi)
library(tidyverse)


In [ ]:
%%R

path <- "/content/drive/MyDrive/PI_JP/"

# PADRONIZACAO E LEITURA
df1 <- read_delim(paste0(path, "datatran2021.csv"), delim = ";", locale = locale(encoding = "latin1")) %>% mutate(across(any_of(c("latitude", "longitude")), as.character))
df2 <- read_delim(paste0(path, "datatran2022.csv"), delim = ";", locale = locale(encoding = "latin1")) %>% mutate(across(any_of(c("latitude", "longitude")), as.character))
df3 <- read_delim(paste0(path, "datatran2023.csv"), delim = ";", locale = locale(encoding = "latin1")) %>% mutate(across(any_of(c("latitude", "longitude")), as.character))
df4 <- read_delim(paste0(path, "datatran2024.csv"), delim = ";", locale = locale(encoding = "latin1")) %>% mutate(across(any_of(c("latitude", "longitude")), as.character))
df5 <- read_delim(paste0(path, "datatran2025.csv"), delim = ";", locale = locale(encoding = "latin1")) %>% mutate(across(any_of(c("latitude", "longitude")), as.character))

# JUNCAO DE TABELAS
df_final <- bind_rows(df1, df2, df3, df4, df5)

# SALVAR TABELA FINAL
write_excel_csv(df_final, paste0(path, "dataset_final.csv"), delim = ";")

# DIMENSAO DA TABELA FINAL
dim(df_final)

In [6]:
%%R
# VERIFICA LINHAS DUPLICADAS

total_original <- nrow(df_final)

total_unicas <- nrow(distinct(df_final))

duplicados <- total_original - total_unicas

print(paste("Registros duplicados encontrados:", duplicados))

[1] "Registros duplicados encontrados: 0"


In [7]:
%%R

# CARREGA TABELA FINAL

dados <- read.csv2("/content/drive/MyDrive/PI_JP/dataset_final.csv", fileEncoding="UTF-8")

# REMOCAO DE ACENTOS
dados_sp <- dados %>%
  filter(uf == "SP") %>%
  mutate(across(where(is.character), ~ stri_trans_general(., "Latin-ASCII")))

# SALVAMENTO USANDO FILE ENCONDING LATIN1 PRA COMTABILIDADE
write.csv2(dados_sp,
           "/content/drive/MyDrive/PI_JP/dataset_final_SP.csv",
           row.names = FALSE,
           fileEncoding = "latin1")

In [ ]:
%%R

# ESCOPO DO DICINARIO DE VARIAVEIS
dicionario <- data.frame(
  Variavel = names(dados),
  Tipo_R = sapply(dados, class),
  Descricao = "",
  Valores = "",
  stringsAsFactors = FALSE
)

# VISUALIZACAO
print(dicionario)

# SALVA NO DRIVE
write.csv2(dicionario,
           "/content/drive/MyDrive/PI_JP/dicionario_variaveis.csv",
           row.names = FALSE,
           fileEncoding = "UTF-8")

In [ ]:
%%R

print(dados_sp)